# DeepFER: Facial Emotion Recognition Using Deep Learning

**Deep Learning for Computer Vision**

This notebook builds, trains, and evaluates a Convolutional Neural Network (CNN) that
classifies facial expressions into **7 emotion classes**: `angry`, `disgust`, `fear`,
`happy`, `neutral`, `sad`, `surprise`.

**Pipeline:**
1. Load dataset (folder-per-class structure)
2. Exploratory data analysis (class balance, sample images)
3. Preprocessing & data augmentation
4. CNN architecture
5. Training with callbacks
6. Evaluation (accuracy, precision/recall/F1, confusion matrix)
7. Save model + label map for deployment (Streamlit app)


## 1. Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)


## 2. Configuration

Update `TRAIN_DIR` and `TEST_DIR` to point at your dataset. Expected structure:

```
dataset/
├── train/
│   ├── angry/
│   ├── disgust/
│   ├── fear/
│   ├── happy/
│   ├── neutral/
│   ├── sad/
│   └── surprise/
└── test/
    ├── angry/
    ├── disgust/
    ├── fear/
    ├── happy/
    ├── neutral/
    ├── sad/
    └── surprise/
```


In [ ]:
TRAIN_DIR = "dataset/train"
TEST_DIR = "dataset/test"

IMG_SIZE = (48, 48)      # classic FER-style input size
BATCH_SIZE = 64
COLOR_MODE = "grayscale" # change to "rgb" if your images are in color
VAL_SPLIT = 0.15
EPOCHS = 50


## 3. Load Dataset

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    label_mode="categorical",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    label_mode="categorical",
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    label_mode="categorical",
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes:", class_names)


## 4. Exploratory Data Analysis

In [ ]:
# Class distribution in the training folder
counts = {c: len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in class_names}
dist_df = pd.DataFrame(list(counts.items()), columns=["emotion", "count"]).sort_values("count", ascending=False)
display(dist_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=dist_df, x="emotion", y="count", hue="emotion", palette="viridis", legend=False)
plt.title("Training Set Class Distribution")
plt.ylabel("Number of images")
plt.xlabel("")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# Sample images per class
plt.figure(figsize=(14, 6))
for i, cname in enumerate(class_names):
    folder = os.path.join(TRAIN_DIR, cname)
    fname = os.listdir(folder)[0]
    img = tf.keras.utils.load_img(os.path.join(folder, fname), color_mode=COLOR_MODE, target_size=IMG_SIZE)
    plt.subplot(1, NUM_CLASSES, i + 1)
    plt.imshow(img, cmap="gray" if COLOR_MODE == "grayscale" else None)
    plt.title(cname)
    plt.axis("off")
plt.tight_layout()
plt.show()


## 5. Preprocessing & Data Augmentation

Pixel values are rescaled to `[0, 1]`. Augmentation (rotation, zoom, horizontal flip) is
applied only during training to improve generalization, matching the augmentation strategy
described in the project deck.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

normalization_layer = layers.Rescaling(1. / 255)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
])

def prepare(ds, training=False):
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

train_ds_prepped = prepare(train_ds, training=True)
val_ds_prepped = prepare(val_ds, training=False)
test_ds_prepped = prepare(test_ds, training=False)


## 6. CNN Architecture

A compact CNN with progressively deeper convolutional blocks (Conv → BatchNorm → ReLU →
MaxPool → Dropout), followed by dense classification layers. This architecture is small
enough to train quickly on 48×48 images while still having enough capacity for 7-class
emotion classification.


In [ ]:
def build_model(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return models.Model(inputs, outputs, name="DeepFER_CNN")

CHANNELS = 1 if COLOR_MODE == "grayscale" else 3
model = build_model((IMG_SIZE[0], IMG_SIZE[1], CHANNELS), NUM_CLASSES)
model.summary()


## 7. Compile & Callbacks

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

os.makedirs("outputs", exist_ok=True)

cb_list = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
    callbacks.ModelCheckpoint("outputs/deepfer_best.keras", monitor="val_accuracy", save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
]


## 8. Train the Model

In [ ]:
history = model.fit(
    train_ds_prepped,
    validation_data=val_ds_prepped,
    epochs=EPOCHS,
    callbacks=cb_list,
)


## 9. Training Curves

In [ ]:
hist_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(hist_df["accuracy"], label="train")
axes[0].plot(hist_df["val_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(hist_df["loss"], label="train")
axes[1].plot(hist_df["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## 10. Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds_prepped)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")


In [ ]:
# Detailed classification report + confusion matrix
y_true = []
y_pred = []

for images, labels in test_ds_prepped:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 11. Save Model & Label Map

Saves the trained model and a `label_map.json` (class index → emotion name), both used
directly by the accompanying Streamlit app (`app.py`).


In [ ]:
model.save("outputs/deepfer_model.keras")

label_map = {i: name for i, name in enumerate(class_names)}
with open("outputs/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

print("Saved: outputs/deepfer_model.keras")
print("Saved: outputs/label_map.json")
print(label_map)


## 12. Sample Predictions

In [ ]:
plt.figure(figsize=(14, 8))
for images, labels in test_ds.take(1):
    preds = model.predict(normalization_layer(images), verbose=0)
    for i in range(min(10, len(images))):
        plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].numpy().astype("uint8").squeeze(), cmap="gray" if COLOR_MODE == "grayscale" else None)
        true_label = class_names[np.argmax(labels[i])]
        pred_label = class_names[np.argmax(preds[i])]
        color = "green" if true_label == pred_label else "red"
        plt.title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=9)
        plt.axis("off")
plt.tight_layout()
plt.show()
